# Kokoroで日英音声ペアを作る

JParaCrawlの対訳をKokoroで読み上げ、Whisperで品質を確認します。進捗と成果物はDriveに保存されます。

| やりたいこと | 実行するセル | ランタイム |
| --- | --- | --- |
| 初回の動作確認 | **1 → 2 → 3**。10件を確認してから100件 | L4 GPU |
| 速度測定と最終テスト | **4 → 5 → 6**。選んだ設定で1000件 | L4 GPU |
| 本番開始・途中再開 | **1 → 2 → 5 → 7 → 8**。保存済みの設定を使う | L4 GPU |
| 全shard完了後の最終集計 | **1 → 2 → 9**。2で `RELEASE_ONLY = True` | CPU可 |

**必要なセルだけ実行してください。** 新しいランタイムでは1から環境を作り直します。完了したテストは再利用できます。途中から続けるときはDriveの入力・設定・保存先を揃えます。

詳しいコマンドと設定は運用ガイドにあります。 [Kokoro運用ガイド](https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus/blob/main/docs/kokoro-runbook.md)


## 1. Driveに接続し、実行環境を準備する

音声を生成する場合は **L4 GPU**、最終集計だけなら **CPU（アクセラレータなし）** を選びます。このセルはDrive接続、`main` の取得、Python 3.12と依存関係の準備を行います。

以前の手動パッチで「未コミット変更」が出たら、`REPO_DIR` の末尾に `-v2` などを付けて実行します。新しく取得するコードのフォルダ名だけが変わり、Driveの保存先は同じです。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json
from pathlib import Path
REPO_DIR = Path('/content/ja-en-direct-s2st-corpus-kokoro')
BRANCH = 'main'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus.git', str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True)
    assert not dirty.strip(), '未コミット変更があります。保存してから再開してください。'
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', 'origin/' + BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'espeak-ng', 'build-essential', 'cmake'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
UV = [sys.executable, '-m', 'uv']
subprocess.run(UV + ['python', 'install', '3.12'], check=True)
ENV_DIR = Path('/content/s2st-kokoro-py312')
if not ENV_DIR.exists():
    subprocess.run(UV + ['venv', '--python', '3.12', '--seed', str(ENV_DIR)], check=True)
PYTHON = str(ENV_DIR / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchaudio==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements/kokoro-colab.txt'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
# UniDicパッケージだけでは辞書本体は入らないため、版を指定して取得
dictionary_check = "from pathlib import Path; import unidic; p=Path(unidic.DICDIR); assert (p/'sys.dic').is_file() and (p/'version').read_text().strip() == 'unidic-3.1.0+2021-08-31'"
if subprocess.run([PYTHON, '-c', dictionary_check]).returncode:
    subprocess.run([PYTHON, '-m', 'unidic', 'download', '3.1.0+2021-08-31'], check=True)
# Qwen用バッチ設定を引き継がない
os.environ.pop('S2ST_TTS_BATCH_SIZE', None)
os.environ.pop('S2ST_ASR_BATCH_SIZE', None)
os.environ['HF_HOME'] = '/content/hf-kokoro-cache'
os.environ['S2ST_INPUT_INDEX_DIR'] = '/content/s2st-input-index'
os.environ['S2ST_MODEL_ROOT'] = '/content/kokoro-models'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('環境の準備が完了しました。次は2の入力確認です。', flush=True)


## 2. 設定を読み込む

**最終集計だけなら `RELEASE_ONLY = True`** に変更し、このセルの次は **9** へ進みます。保存済みの設定を使い、GPU確認とモデルのダウンロードを省きます。

初回・生成再開は `RELEASE_ONLY = False` で実行します。入力の件数とSHA256、GPUと空き容量を確認します。初回は先に `jparacrawl-filtering/outputs/text-corpus/` に `manifest.jsonl` と `summary.json` を用意してください。

設定はDriveの `ja-en-direct-s2st-corpus/kokoro-jparacrawl-l4-config.yaml`、ログは同じ場所の `kokoro-logs/` に保存します。


In [ ]:
RELEASE_ONLY = False  # 最終集計だけならTrue

import shutil
DATA_ROOT = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = DATA_ROOT / 'kokoro-jparacrawl-l4-config.yaml'
if not CONFIG.exists():
    if RELEASE_ONLY:
        raise FileNotFoundError(f'本番で使用した設定ファイルが見つかりません: {CONFIG}')
    shutil.copyfile(REPO_DIR / 'configs/production-kokoro-jparacrawl-l4.yaml', CONFIG)
print(CONFIG.read_text())
# 保存済み設定からパスを取得（ホストPythonのyamlには依存しない）
def read_settings():
    script = 'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))'
    return json.loads(subprocess.check_output([PYTHON, '-c', script, str(CONFIG)], text=True))
settings = read_settings()
for key in ('manifest', 'summary'):
    path = Path(settings['prepare'][key])
    assert path.is_file(), f'export入力がありません: {path}'
    print(key, path, path.stat().st_size)
# 実環境の依存・実装commitも保存
ENV_REPORT = DATA_ROOT / 'kokoro-environment'
ENV_REPORT.mkdir(exist_ok=True)
(ENV_REPORT / 'pip-freeze.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
(ENV_REPORT / 'git-commit.txt').write_text(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True))

from datetime import datetime
LOG_DIR = DATA_ROOT / 'kokoro-logs'
LOG_DIR.mkdir(exist_ok=True)
def run_command(command, *args, production=False):
    cmd = [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG)]
    if production:
        cmd.append('--allow-production')
    cmd.extend([command, *map(str, args)])
    log_path = LOG_DIR / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + command + '.log')
    print('Log:', log_path, flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                rc = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if rc:
        raise RuntimeError(f'{command} failed ({rc}): {log_path}')

if RELEASE_ONLY:
    print('最終集計の準備が完了しました。セル9へ進んでください。', flush=True)
else:
    print('入力データを検証します。', flush=True)
    run_command('prepare')
    print('GPUと空き容量を確認します。', flush=True)
    run_command('preflight')
    print('音声モデル・辞書・声を準備します。', flush=True)
    run_command('preload')
    print('入力とGPUの確認が完了しました。次は3の10件テストです。', flush=True)


## 3. 10件と100件で動作を確認する

まず `SMOKE_PAIRS = 10` で実行し、合格した日英音声と採用数を確認します。次に `SMOKE_PAIRS = 100` に変えて同じセルを実行します。件数別に結果を保存し、不採用理由は `all.jsonl` に残ります。

声の候補を聴くときだけ `RUN_VOICE_SAMPLES = True` にします。使用する声の変更はDriveの設定ファイルで行います。


In [ ]:
SMOKE_PAIRS = 10  # 最初は10。結果を確認してから100に変えて再実行
RUN_SMOKE = True  # テスト済みで再実行しないときはFalse
RUN_VOICE_SAMPLES = False  # 声の候補も聴きたい場合だけTrue

VOICE_DIR = DATA_ROOT / 'kokoro-voice-samples'
if RUN_VOICE_SAMPLES:
    run_command('voice-samples', '--output-dir', VOICE_DIR, '--en-voices', 'af_heart', 'af_bella', 'am_adam')
    from IPython.display import Audio, display
    for path in sorted(VOICE_DIR.glob('*.wav')):
        print(path.name)
        display(Audio(filename=str(path)))

if RUN_SMOKE:
    assert SMOKE_PAIRS in (10, 100), 'このセルは10件または100件用です。1000件は6で実行します。'
    run_command('smoke-test', '--pairs', SMOKE_PAIRS)
    settings = read_settings()
    release_dir = Path(settings['smoke']['output_dir']) / f'pairs-{SMOKE_PAIRS}/production/manifests/releases'
    summary = json.loads((release_dir / 'summary.json').read_text(encoding='utf-8'))
    print('結果フォルダ:', release_dir)
    keys = ['total_pairs', 'accepted_pairs', 'rejected_pairs', 'acceptance_rate', 'failures', 'tts_rtf_total']
    print(json.dumps({key: summary.get(key) for key in keys}, ensure_ascii=False, indent=2))
    with (release_dir / 'accepted.jsonl').open(encoding='utf-8') as manifest:
        first_line = manifest.readline()
    if first_line:
        from IPython.display import Audio, display
        pair = json.loads(first_line)
        for lang in ('ja', 'en'):
            print(lang, pair[f'{lang}_text'])
            display(Audio(filename=pair[f'{lang}_wav_16k']))
    else:
        print('採用ペアがありません。all.jsonlで不採用理由を確認してください。')
else:
    print('少量テストはスキップしました。')


## 4. L4で処理速度を比較する

100件テストを確認したら `RUN_L4_TUNING = True` にします。同じ200ペアを使い、Kokoroの並列数（1 / 2 / 4 / 6）とWhisperのバッチ数（8 / 16 / 24 / 32）を順に比較します。

通常は `TUNING_STAGE = 'all'` を使います。Whisperだけ測り直す `'asr'` は、同じランタイムに保存された測定用音声と、同じコードのcommitが必要です。実配分・速度・VRAM・品質比較はレポートに残ります。


In [ ]:
RUN_L4_TUNING = False  # 100件テストを確認してからTrue
TUNING_STAGE = 'all'  # 'all': 両方 / 'tts': 音声生成だけ / 'asr': 文字起こしだけ

if RUN_L4_TUNING:
    assert TUNING_STAGE in ('all', 'tts', 'asr')
    run_command('preflight')
    run_command('benchmark-performance', '--stage', TUNING_STAGE)
else:
    print('速度測定はスキップしました。保存済みのレポートがあれば表示します。')

PERFORMANCE_REPORT = Path(read_settings().get('performance', {}).get('report_dir', str(DATA_ROOT / 'kokoro-performance'))) / 'l4-performance.json'
if PERFORMANCE_REPORT.is_file():
    performance_report = json.loads(PERFORMANCE_REPORT.read_text())
    print('GPU:', performance_report.get('gpu'))
    print('推奨する音声生成の並列数:', performance_report.get('recommended_tts_workers'))
    print('推奨する文字起こしのバッチ数:', performance_report.get('recommended_asr_batch_size'))
    for stage in ('tts_candidates', 'asr_candidates'):
        print('音声生成の比較:' if stage == 'tts_candidates' else '文字起こしの比較:')
        for candidate in performance_report.get(stage, []):
            keys = ['worker_count', 'batch_size', 'status', 'audio_hours_per_wall_hour', 'audio_seconds_per_wall_second', 'headroom_warning', 'quality_equivalent', 'gpu_peak_used_gib_sampled', 'peak_reserved_vram_gib_reference', 'error']
            print({key: candidate.get(key) for key in keys})
else:
    print('速度測定のレポートはまだありません。')

print('audio_hours_per_wall_hour / audio_seconds_per_wall_second: 大きいほど高速')
print('quality_equivalent: 品質比較の一致 / headroom_warning: VRAMの余裕が少ない候補')
print('gpu_peak_used_gib_sampled: GPU全体の使用量を5秒ごとに測定した最大値')
print('peak_reserved_vram_gib_reference: プロセスごとの参考値。GPU全体の実測ピークとは異なります。')


## 5. 採用する設定を保存する

速度測定の結果を見て、Driveの設定ファイルに **`tts.worker_count`** と **`asr.batch_size`** を保存します。このセルで表示した値を確認し、`PERFORMANCE_CONFIG_CONFIRMED = True` にします。

本番の再開時は保存済みの設定を確認します。声・モデル・Whisperのバッチ数を変えて再試験する場合は、`smoke.output_dir` を別の保存先にします。


In [ ]:
settings = read_settings()
print('Config:', CONFIG)
print('Kokoro workers:', settings['tts'].get('worker_count', 1))
print('Whisper batch:', settings['asr']['batch_size'])
PERFORMANCE_CONFIG_CONFIRMED = False
print('設定の確認:', '確認済み。6へ進めます。' if PERFORMANCE_CONFIG_CONFIRMED else '未確認。表示された設定を確認してください。')


## 6. 採用した設定で1000件を試す

5で設定を確認してから、`RUN_BENCHMARK_1000 = True` にして実行します。学習用の1000ペアを文の長さ別に抽出し、音声生成と品質検査を行います。既定では不足する長さ区分の分を他の区分へ振り分け、実配分を記録します。

採用率、CER/WER、長文の失敗、処理時間を確認します。結果は `smoke.output_dir/pairs-1000/production/manifests/releases/` に保存します。


In [ ]:
RUN_BENCHMARK_1000 = False
BENCHMARK_SHORTFALL = 'redistribute'  # 不足分を再配分。固定配分を必須にする場合は'error'
if RUN_BENCHMARK_1000:
    assert PERFORMANCE_CONFIG_CONFIRMED, '推奨値を確認し、設定を手動で確定してから実行してください。'
    run_command('smoke-test', '--pairs', 1000, '--benchmark-seed', 42,
                '--benchmark-shortfall', BENCHMARK_SHORTFALL)
    settings = read_settings()
    report_path = Path(settings['smoke']['output_dir']) / 'pairs-1000/production/manifests/releases/summary.json'
    report = json.loads(report_path.read_text())
    keys = ['total_pairs', 'length_bin_distribution', 'accepted_pairs', 'rejected_pairs', 'ja_hours', 'en_hours', 'attempt_metrics', 'failures', 'duration_failures_by_length_bin', 'duration_failures_by_language_and_length_bin', 'tts_rtf_total', 'asr_phase_wall_seconds', 'asr_inference_seconds', 'full_pipeline_wall_seconds', 'checkpoint_writes', 'checkpoint_write_seconds', 'directory_scan_count', 'directory_scan_seconds', 'train', 'dev', 'test']
    view = {key: report.get(key) for key in keys}
    view['gpu'] = [i.get('gpu') for i in report['invocations']]
    view['parent_peak_allocated_vram_gib'] = max((i.get('peak_allocated_vram_gib', 0) for i in report['invocations']), default=0)
    view['parent_peak_reserved_vram_gib'] = max((i.get('peak_reserved_vram_gib', 0) for i in report['invocations']), default=0)
    view['tts_worker_per_process_peaks'] = [i.get('tts_worker_per_process_peak', {}) for i in report['invocations']]
    view['phase_throughput'] = [{key: i.get(key) for key in ('kokoro_worker_count', 'asr_batch_size', 'tts_generated_utterances', 'tts_generated_audio_seconds', 'tts_phase_seconds', 'tts_audio_hours_per_wall_hour', 'asr_audio_seconds', 'asr_phase_seconds')} for i in report['invocations']]
    print(json.dumps(view, ensure_ascii=False, indent=2))
else:
    print('1000件テストは無効です。5の設定確認後に有効化してください。')


## 7. 本番の進捗を確認する

`completed_shards` は完了数、`remaining_shards` は残り、`next_shard` は次の処理対象です。**残りが0なら9の最終集計へ**進みます。


In [ ]:
run_command('plan')
run_command('status')


## 8. 本番の音声生成と品質検査を実行する

1000件テストを確認し、本番を始めるときに `APPROVE_PRODUCTION = True` にします。最初は `RUN_ALL_REMAINING = False` で次の1 shardを実行し、結果を確認してから `True` に変更すると残りをすべて処理します。

`completed=128/128 remaining=0` が出てセルが終了したら、**9で最終集計**します。音声と進捗はDriveの `run.output_dir` に残ります。同じ保存先への書き込みは1つのランタイムで行ってください。


In [ ]:
APPROVE_PRODUCTION = False  # 1000件の結果を確認し、本番を始める場合だけTrue
RUN_ALL_REMAINING = False  # False: 次の1 shard / True: 残りすべて

if APPROVE_PRODUCTION:
    assert PERFORMANCE_CONFIG_CONFIRMED, '5で使用する設定を確認してください。'
    if RUN_ALL_REMAINING:
        run_command('run-remaining-shards', '--skip-consolidate', production=True)
    else:
        run_command('run-next-shard', production=True)
else:
    print('本番は無効です。1000件テストの結果を確認してから有効化してください。')


## 9. 完成データを書き出す（CPU可）

全shardの完了後、`WRITE_RELEASE = True` にして実行します。書き出し先は `run.output_dir/manifests/releases/` です。

| 検証方法 | 検査内容 |
| --- | --- |
| `'inventory'` | WAVの存在とQC記録を確認。各WAVの形式・中身・SHA256の再照合を省く |
| `'full'`（既定） | WAVを開き、形式・長さ・SHA256まで照合 |

Drive上で高速に集計したい場合は `RELEASE_AUDIO_VERIFICATION = 'inventory'` を選びます。使用した方法は `summary.json` の `audio_verification` に記録されます。

`summary.json` の **`complete: true`** と **`missing_shards: []`** を確認します。下流の学習にはtrainとdevが入った **`accepted.jsonl`** を使います。


In [ ]:
WRITE_RELEASE = False  # 全shardの完了を確認したらTrue
RELEASE_AUDIO_VERIFICATION = 'full'  # 'inventory' でファイルの存在とQC記録を確認
if WRITE_RELEASE:
    run_command('consolidate', '--audio-verification', RELEASE_AUDIO_VERIFICATION)
    release_dir = Path(read_settings()['run']['output_dir']) / 'manifests/releases'
    summary = json.loads((release_dir / 'summary.json').read_text(encoding='utf-8'))
    print('完成データの保存先:', release_dir)
    print('下流に渡すmanifest:', release_dir / 'accepted.jsonl')
    keys = ('complete', 'missing_shards', 'total_pairs', 'accepted_pairs', 'rejected_pairs', 'audio_verification')
    print({key: summary.get(key) for key in keys})
else:
    print('本番データの書き出しは無効です。テスト結果は各pairs-Nフォルダにあります。')
